In [ ]:
from pathlib import Path
import sys


def find_repo_root(start_dir: Path) -> Path:
    """Find the project root containing data/raw and classification/svm."""
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / "data" / "raw").exists() and (candidate / "classification" / "svm").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find project root containing data/raw and classification/svm from the current working directory."
    )


repo_root = find_repo_root(Path.cwd())

preprocess_dir = repo_root / "classification" / "svm"
if not (preprocess_dir / "preprocess_pipeline.py").exists():
    raise FileNotFoundError(f"Missing preprocess module at: {preprocess_dir / 'preprocess_pipeline.py'}")
if str(preprocess_dir) not in sys.path:
    sys.path.insert(0, str(preprocess_dir))

from preprocess_pipeline import preprocess_excel_file

TEXT_COL = "body_display"
INPUT_PATH = repo_root / "data" / "raw" / "singapore_wp_comments_display.xlsx"
OUTPUT_PATH = repo_root / "data" / "cleaned" / "singapore_wp_comments_display_clean.xlsx"

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Missing input file: {INPUT_PATH}")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df = preprocess_excel_file(INPUT_PATH, text_col=TEXT_COL, output_path=OUTPUT_PATH)
print(df[[TEXT_COL, "clean_text"]].head(10))
print(f"Saved -> {OUTPUT_PATH}")

                                        body_display  \
0  I think it applies for social media in general...   
1  I’m worried that WP is getting over reliant on...   
2  Singapore is too small. You go to where your c...   
3  The problem is that GST reduction is not the o...   
4  Out of all the mosquitoes maybe Hazel Poa and ...   
5  Nah.. Ariffin Sha may seem like a good speaker...   
6  WP won't take mosquitos. They worked so hard t...   
7  Unless WP wants to take the mosquitoes under t...   
8  It's not a given for WP to win the by-election...   
9  I think SDP is too far to the left for fiscall...   

                                          clean_text  
0  i think it applies for social media in general...  
1  i ’ m worried that workers party is getting ov...  
2  singapore is too small. you go to where your c...  
3  the problem is that gst reduction is not the o...  
4  out of all the mosquitoes maybe hazel poa and ...  
5  nah.. ariffin sha may seem like a good speaker... 

In [2]:
"""
Two-stage sentiment pipeline for Singapore political Reddit data.

Flow:
1) Load preprocessed input from Cell 1 output
2) Stage 1: subjectivity filter (objective -> Neutral)
3) Stage 2: SVM sentiment classification (subjective only)
4) Entity/aspect enrichment and optional sarcasm soft-adjustment
5) Save CSV and JSON outputs
"""

import re
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

# ===============================
# SECTION 0 - OPTIONAL BACKENDS
# ===============================
# spaCy and NLTK are optional. The pipeline falls back safely if unavailable.
SPACY_NLP = None
NLTK_LESK = None

try:
    import spacy
    SPACY_NLP = spacy.load("en_core_web_sm")
    print("[setup] spaCy loaded (en_core_web_sm)")
except Exception:
    print("[setup] spaCy not available - using regex-only NER")

try:
    from nltk.wsd import lesk as _lesk
    NLTK_LESK = _lesk
    print("[setup] NLTK lesk loaded")
except Exception:
    print("[setup] NLTK lesk not available - using default aspect mapping")


# =====================================
# SECTION 1 - LOAD DATA AND ARTIFACTS
# =====================================
def find_repo_root(start_dir: Path) -> Path:
    """Find project root containing data/cleaned and classification/svm."""
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / "data" / "cleaned").exists() and (candidate / "classification" / "svm").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find project root containing data/cleaned and classification/svm from the current working directory."
    )


repo_root = find_repo_root(Path.cwd())
DATA_PATH = repo_root / "data" / "cleaned" / "singapore_wp_comments_display_clean.xlsx"
TEXT_COL = "clean_text"
RAW_COL = "body_display"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find cleaned inference input: {DATA_PATH}. Run notebook Cell 1 first."
    )

df = pd.read_excel(DATA_PATH)

if RAW_COL not in df.columns:
    raise ValueError(f"Missing required column: '{RAW_COL}'")

# Fallback for datasets where clean_text is absent.
if TEXT_COL not in df.columns:
    df[TEXT_COL] = df[RAW_COL]
    print(f"[warn] '{TEXT_COL}' not found - using '{RAW_COL}' as fallback")

df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
df[RAW_COL] = df[RAW_COL].fillna("").astype(str)

print(f"[load] Dataset: {len(df):,} rows from {DATA_PATH}")

ARTIFACTS_DIR = repo_root / "classification" / "svm" / "model_artifacts"
for pkl_name, pkl_path, var_name in [
    ("TF-IDF", ARTIFACTS_DIR / "tfidf.pkl", "tfidf_vectorizer"),
    ("SVM model", ARTIFACTS_DIR / "svm_model.pkl", "svm_model"),
]:
    if not pkl_path.exists():
        raise FileNotFoundError(
            f"Missing {pkl_name} pickle at {pkl_path}. Run notebook Step 2 first."
        )
    with open(pkl_path, "rb") as f:
        globals()[var_name] = pickle.load(f)
    print(f"[load] {pkl_name} loaded from {pkl_path}")


# ============================
# SECTION 2 - KNOWLEDGE BASE
# ============================
PARTY_ALIAS_TO_CANONICAL = {
    "wp": "Workers Party",
    "workers party": "Workers Party",
    "worker party": "Workers Party",
    "workers' party": "Workers Party",
    "pap": "Peoples Action Party",
    "people action party": "Peoples Action Party",
    "people's action party": "Peoples Action Party",
    "psp": "Progress Singapore Party",
    "progress singapore party": "Progress Singapore Party",
    "sdp": "Singapore Democratic Party",
    "singapore democratic party": "Singapore Democratic Party",
    "rp": "Reform Party",
    "reform party": "Reform Party",
    "rdu": "Red Dot United",
    "red dot united": "Red Dot United",
}

PERSON_ALIAS_TO_CANONICAL = {
    "pritam": "Pritam Singh",
    "ps": "Pritam Singh",
    "pritam singh": "Pritam Singh",
    "sylvia": "Sylvia Lim",
    "sylvia lim": "Sylvia Lim",
    "jamus": "Jamus Lim",
    "jamus lim": "Jamus Lim",
    "he ting ru": "He Ting Ru",
    "ting ru": "He Ting Ru",
    "gerald": "Gerald Giam",
    "gerald giam": "Gerald Giam",
    "lee hsien loong": "Lee Hsien Loong",
    "lhl": "Lee Hsien Loong",
    "pm lee": "Lee Hsien Loong",
    "lee kuan yew": "Lee Kuan Yew",
    "lky": "Lee Kuan Yew",
    "lawrence wong": "Lawrence Wong",
    "lw": "Lawrence Wong",
    "pm wong": "Lawrence Wong",
    "vivian": "Vivian Balakrishnan",
    "vivian balakrishnan": "Vivian Balakrishnan",
    "chan chun sing": "Chan Chun Sing",
    "ccs": "Chan Chun Sing",
    "ong ye kung": "Ong Ye Kung",
    "oyk": "Ong Ye Kung",
}

PERSON_TO_PARTY = {
    "Pritam Singh": "Workers Party",
    "Sylvia Lim": "Workers Party",
    "Jamus Lim": "Workers Party",
    "He Ting Ru": "Workers Party",
    "Gerald Giam": "Workers Party",
    "Lee Hsien Loong": "Peoples Action Party",
    "Lee Kuan Yew": "Peoples Action Party",
    "Lawrence Wong": "Peoples Action Party",
    "Vivian Balakrishnan": "Peoples Action Party",
    "Chan Chun Sing": "Peoples Action Party",
    "Ong Ye Kung": "Peoples Action Party",
}

ASPECTS = {
    "economy": [
        "economy", "economic", "inflation", "cost", "price", "gst", "tax",
        "cost of living", "jobs", "salary", "wage", "business", "foreign policy",
    ],
    "housing": [
        "hdb", "housing", "rent", "property", "bto", "mortgage", "home", "flat",
    ],
    "government": [
        "policy", "minister", "government", "parliament", "leadership", "governance",
    ],
}

# Ambiguous tokens that may map to multiple aspects.
AMBIGUOUS_TERMS = {
    "policy": "government",
    "cost": "economy",
    "property": "housing",
}

PRONOUN_PERSON = {"he", "him", "his", "she", "her", "hers"}
PRONOUN_GROUP = {"they", "them", "their", "theirs"}

SARCASTIC_MARKERS = {
    "yeah right", "as if", "sure", "totally", "obviously", "/s", "lol", "lmao"
}

SUBJECTIVE_CUES = {
    "i think", "i feel", "i guess", "in my opinion", "seems",
    "looks like", "probably",
}

# Texts below this score are treated as objective and set to Neutral.
SUBJECTIVITY_THRESHOLD = 0.05


# ============================
# SECTION 3 - TEXT UTILITIES
# ============================
def normalize_text(text: str) -> str:
    """Lowercase and normalize text for matching."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-']", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def _compile_term_pattern(term: str, allow_hyphen_suffix: bool = False) -> re.Pattern:
    """Create a whole-word regex for a term."""
    term_norm = normalize_text(term)
    if not term_norm:
        return re.compile(r"a^")
    phrase = re.escape(term_norm).replace(r"\ ", r"\s+")
    suffix = r"(?:-[a-z0-9]+)*" if allow_hyphen_suffix else ""
    return re.compile(
        rf"(?<![a-z0-9]){phrase}{suffix}(?![a-z0-9])",
        flags=re.IGNORECASE,
    )


def _unique_preserve_order(items: list) -> list:
    """Deduplicate while keeping first-seen order."""
    return list(dict.fromkeys(items))


def split_sentences(text: str) -> list[str]:
    """Split text into sentences via spaCy if available, regex otherwise."""
    if not text:
        return []
    if SPACY_NLP is not None:
        sents = [s.text.strip() for s in SPACY_NLP(text).sents if s.text.strip()]
        return sents or [text]
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p for p in parts if p]


# Compile once to avoid repeated regex compilation in loops.
PERSON_PATTERNS = {
    alias: _compile_term_pattern(alias)
    for alias in PERSON_ALIAS_TO_CANONICAL
}
PARTY_PATTERNS = {
    alias: _compile_term_pattern(
        alias,
        allow_hyphen_suffix=(alias.isupper() and alias.isalpha()),
    )
    for alias in PARTY_ALIAS_TO_CANONICAL
}
ASPECT_PATTERNS = {
    aspect: [_compile_term_pattern(kw) for kw in keywords]
    for aspect, keywords in ASPECTS.items()
}


# ========================================
# SECTION 4 - STAGE 1 SUBJECTIVITY FILTER
# ========================================
def subjectivity_score(text: str) -> float:
    """Compute a heuristic subjectivity score in [0, 1]."""
    t = text.lower()
    score = 0.4 if any(cue in t for cue in SUBJECTIVE_CUES) else 0.0
    pronouns = len(re.findall(r"\b(i|me|my|we|our)\b", t))
    score += min(0.6, pronouns * 0.1)
    return round(min(score, 1.0), 4)


def is_subjective(text: str) -> bool:
    """Return True when text passes the subjectivity threshold."""
    return subjectivity_score(text) >= SUBJECTIVITY_THRESHOLD


# ======================================
# SECTION 5 - STAGE 2 SVM CLASSIFICATION
# ======================================
def classify_sentiment_batch(texts: list[str]) -> tuple[list[str], list[float]]:
    """Predict sentiment labels and confidence scores for a batch of texts."""
    if not texts:
        return [], []

    safe = [t if isinstance(t, str) else "" for t in texts]
    X = tfidf_vectorizer.transform(safe)
    preds = svm_model.predict(X)

    if hasattr(svm_model, "predict_proba"):
        probs = svm_model.predict_proba(X)
        confidences = probs.max(axis=1).astype(float)
    elif hasattr(svm_model, "decision_function"):
        margins = np.asarray(svm_model.decision_function(X))
        raw = margins.max(axis=1) if margins.ndim > 1 else margins.astype(float)
        confidences = 1.0 / (1.0 + np.exp(-raw))
    else:
        confidences = np.ones(len(safe), dtype=float)

    return [str(p) for p in preds], [round(float(c), 4) for c in confidences]


def classify_sentiment(text: str) -> tuple[str, float]:
    """Single-text wrapper for classify_sentiment_batch."""
    labels, scores = classify_sentiment_batch([text])
    return (labels[0], scores[0]) if labels else ("Neutral", 0.0)


# ===============================
# SECTION 6 - FEATURE ENRICHMENT
# ===============================
def disambiguate_aspect_term(term: str, sentence: str) -> str:
    """Resolve ambiguous aspect terms with NLTK lesk when available."""
    default = AMBIGUOUS_TERMS.get(term.lower())
    if default is None:
        return None
    if NLTK_LESK is None:
        return default

    try:
        context = sentence.lower().split()
        syn = NLTK_LESK(context, term.lower())
        if syn is None:
            return default
        lexname = syn.lexname()
        if term.lower() == "property" and ("possession" in lexname or "location" in lexname):
            return "housing"
        if term.lower() == "policy" and ("cognition" in lexname or "communication" in lexname):
            return "government"
        if term.lower() == "cost" and ("quantity" in lexname or "possession" in lexname):
            return "economy"
    except Exception:
        return default
    return default


def _regex_entity_matches(text_norm: str, patterns: dict, canonical_map: dict) -> list[str]:
    """Match aliases with regex and return canonical names in text order."""
    hits = []
    for alias, pat in patterns.items():
        m = pat.search(text_norm)
        if m:
            hits.append((m.start(), canonical_map[alias]))
    hits.sort(key=lambda x: x[0])
    return [name for _, name in hits]


def _ner_entity_matches(sentence: str) -> tuple[list[str], list[str]]:
    """Supplement regex extraction with spaCy PERSON/ORG/GPE entities."""
    if SPACY_NLP is None or not sentence:
        return [], []
    doc = SPACY_NLP(sentence)
    persons = [e.text for e in doc.ents if e.label_ == "PERSON"]
    orgs = [e.text for e in doc.ents if e.label_ in {"ORG", "GPE"}]
    return persons, orgs


def _canonicalise_ner(ner_values: list[str], alias_map: dict) -> list[str]:
    """Map raw NER strings to canonical names via alias lookup."""
    out = []
    for val in ner_values:
        v_norm = normalize_text(val)
        if v_norm in alias_map:
            out.append(alias_map[v_norm])
            continue
        for alias, canonical in alias_map.items():
            if alias in v_norm or v_norm in alias:
                out.append(canonical)
                break
    return out


def extract_entities_and_aspects(clean_text: str) -> tuple[list, list, list]:
    """Extract persons, parties, and aspects from clean text."""
    if not clean_text or not clean_text.strip():
        return [], [], []

    sentences = split_sentences(clean_text)
    all_persons, all_parties, all_aspects = [], [], []

    # Track recent entities for simple pronoun backfill across sentences.
    last_person = None
    last_party = None

    for sent in sentences:
        sent_norm = normalize_text(sent)

        persons = _regex_entity_matches(sent_norm, PERSON_PATTERNS, PERSON_ALIAS_TO_CANONICAL)
        parties = _regex_entity_matches(sent_norm, PARTY_PATTERNS, PARTY_ALIAS_TO_CANONICAL)

        ner_p, ner_o = _ner_entity_matches(sent)
        persons += _canonicalise_ner(ner_p, PERSON_ALIAS_TO_CANONICAL)
        parties += _canonicalise_ner(ner_o, PARTY_ALIAS_TO_CANONICAL)

        persons = _unique_preserve_order(persons)
        parties = _unique_preserve_order(parties)

        for person in persons:
            party = PERSON_TO_PARTY.get(person)
            if party:
                parties.append(party)
        parties = _unique_preserve_order(parties)

        tokens = set(sent_norm.split())
        if not persons and any(p in tokens for p in PRONOUN_PERSON) and last_person:
            persons.append(last_person)
            inferred = PERSON_TO_PARTY.get(last_person)
            if inferred:
                parties.append(inferred)
        if not parties and any(p in tokens for p in PRONOUN_GROUP) and last_party:
            parties.append(last_party)

        persons = _unique_preserve_order(persons)
        parties = _unique_preserve_order(parties)

        aspects = []
        for aspect, patterns in ASPECT_PATTERNS.items():
            if any(p.search(sent_norm) for p in patterns):
                aspects.append(aspect)
        for term in AMBIGUOUS_TERMS:
            if _compile_term_pattern(term).search(sent_norm):
                resolved = disambiguate_aspect_term(term, sent_norm)
                if resolved:
                    aspects.append(resolved)
        aspects = _unique_preserve_order(aspects)

        all_persons.extend(persons)
        all_parties.extend(parties)
        all_aspects.extend(aspects)

        if persons:
            last_person = persons[-1]
        if parties:
            last_party = parties[-1]

    return (
        _unique_preserve_order(all_persons),
        _unique_preserve_order(all_parties),
        _unique_preserve_order(all_aspects),
    )


def detect_sarcasm(text: str) -> bool:
    """Heuristic sarcasm detector for low-confidence soft adjustment."""
    t = text.lower()
    if any(marker in t for marker in SARCASTIC_MARKERS):
        return True
    if t.count("!") >= 3 or t.count("?") >= 3:
        return True
    if re.search(r'"[^"]+"', text):
        return True
    return False


# =====================================
# SECTION 7 - MAIN PIPELINE ORCHESTRATOR
# =====================================
def process_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    """Run subjectivity filter, SVM inference, enrichment, and return results."""
    texts = df[TEXT_COL].tolist()
    n = len(texts)
    print(f"\n[pipeline] Processing {n:,} texts ...")

    print("[stage 1] Computing subjectivity scores ...")
    subj_scores = [subjectivity_score(t) for t in texts]
    is_subj_mask = [s >= SUBJECTIVITY_THRESHOLD for s in subj_scores]

    objective_count = sum(1 for v in is_subj_mask if not v)
    subjective_count = sum(is_subj_mask)
    print(
        f"[stage 1] Objective (-> Neutral): {objective_count:,}  |  "
        f"Subjective (-> SVM): {subjective_count:,}"
    )

    # Initialize with Neutral; overwrite subjective rows after SVM inference.
    sentiment_labels = ["Neutral"] * n
    sentiment_scores = [0.0] * n

    print("[stage 2] Classifying subjective texts with SVM ...")
    subjective_indices = [i for i, v in enumerate(is_subj_mask) if v]
    subjective_texts = [texts[i] for i in subjective_indices]

    if subjective_texts:
        svm_labels, svm_conf = classify_sentiment_batch(subjective_texts)
        for idx, label, score in zip(subjective_indices, svm_labels, svm_conf):
            sentiment_labels[idx] = label
            sentiment_scores[idx] = score

    print("[enrich] Extracting entities and aspects ...")
    persons_col = []
    parties_col = []
    aspects_col = []

    for txt in texts:
        persons, parties, aspects = extract_entities_and_aspects(txt)
        persons_col.append(persons)
        parties_col.append(parties)
        aspects_col.append(aspects)

    # Soft adjustment: reduce confidence for likely sarcastic low-confidence rows.
    print("[enrich] Applying sarcasm adjustment ...")
    sarcasm_flags = [detect_sarcasm(t) for t in texts]

    adjusted = 0
    for i in range(n):
        if sarcasm_flags[i]:
            if sentiment_scores[i] < 0.5:
                sentiment_scores[i] = round(sentiment_scores[i] * 0.7, 4)
                adjusted += 1
            elif 0.5 <= sentiment_scores[i] < 0.6:
                sentiment_scores[i] = round(sentiment_scores[i] * 0.85, 4)
                adjusted += 1

    print(f"[enrich] Sarcasm-adjusted labels (soft): {adjusted:,}")

    results = pd.DataFrame({
        "text": df[RAW_COL].values,
        "clean_text": df[TEXT_COL].values,
        "sentiment": sentiment_labels,
        "sentiment_score": sentiment_scores,
        "party": parties_col,
        "person": persons_col,
        "aspect": aspects_col,
    })

    print("\n[pipeline] Sentiment distribution:")
    print(results["sentiment"].value_counts().to_string())
    print(f"\n[pipeline] spaCy NER enabled : {SPACY_NLP is not None}")
    print(f"[pipeline] NLTK lesk enabled : {NLTK_LESK is not None}")

    return results


# ===========================
# SECTION 8 - RUN AND EXPORT
# ===========================
if __name__ == "__main__":
    df_results = process_pipeline(df)

    OUTPUT_DIR = repo_root / "data" / "cleaned"
    OUTPUT_CSV = OUTPUT_DIR / "singapore_wp_comments_display_inference.csv"
    OUTPUT_JSON = OUTPUT_DIR / "singapore_wp_comments_display_inference.json"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    df_results.to_csv(OUTPUT_CSV, index=False)
    df_results.to_json(
        OUTPUT_JSON,
        orient="records",
        indent=2,
        force_ascii=False,
    )

    print(f"\n[done] Saved {OUTPUT_CSV} and {OUTPUT_JSON}")
    print(f"[done] Total rows : {len(df_results):,}")
    print(df_results.head(10).to_string())

[setup] spaCy loaded  (en_core_web_sm)
[setup] NLTK lesk loaded
[load] Dataset: 13,214 rows from /Users/dylan/Documents/GitHub/sc4021-ir/data/cleaned/singapore_wp_comments_display_clean.xlsx
[load] TF-IDF loaded from /Users/dylan/Documents/GitHub/sc4021-ir/classification/svm/model_artifacts/tfidf.pkl
[load] SVM model loaded from /Users/dylan/Documents/GitHub/sc4021-ir/classification/svm/model_artifacts/svm_model.pkl

[pipeline] Processing 13,214 texts …
[stage 1] Computing subjectivity scores …
[stage 1] Objective (→ Neutral): 5,489  |  Subjective (→ SVM):  7,725
[stage 2] Classifying subjective texts with SVM …
[enrich] Extracting entities and aspects …
[enrich] Applying sarcasm adjustment …
[enrich] Sarcasm-adjusted labels (soft): 3,061

[pipeline] Sentiment distribution:
sentiment
Neutral     6986
Positive    5946
Negative     282

[pipeline] spaCy NER enabled : True
[pipeline] NLTK lesk enabled  : True

[done] Saved /Users/dylan/Documents/GitHub/sc4021-ir/data/cleaned/singapore_wp_